# Metadata Loader - Sector-Agnostic Taxonomy

**Purpose**: Load governed sector, role, and skill taxonomies from metadata files instead of hardcoded dictionaries.

**Architecture**:
```
Sector
  ↓
Role Family
  ↓
Canonical Role
  ↓
Skill Category
  ↓
Canonical Skill
```

**This replaces** hardcoded dictionaries in:
* `inter_sector_normalize` (tech-centric NAICS samples)
* `inter_role_map` (tech-centric role dictionary)
* `inter_skill_catalog_sync` (tech-centric skill lists)

**Key Principle**: Technology is just one sector among many. The platform is sector-agnostic.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Metadata file paths
METADATA_BASE = "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata"

CONFIG = {
    "sectors_file": f"{METADATA_BASE}/sectors.csv",
    "role_families_file": f"{METADATA_BASE}/role_families.csv",
    "canonical_roles_file": f"{METADATA_BASE}/canonical_roles.csv",
    "canonical_skills_file": f"{METADATA_BASE}/canonical_skills.csv"
}

print("Metadata Loader Configuration:")
for key, path in CONFIG.items():
    print(f"  {key}: {path}")

In [0]:
# Load sector taxonomy
sectors_df = spark.read.csv(
    CONFIG["sectors_file"],
    header=True,
    inferSchema=True
)

print(f"Loaded {sectors_df.count()} sectors")
print("\nSector Hierarchy:")
display(sectors_df.orderBy("sector_key"))

In [0]:
# Load role families
role_families_df = spark.read.csv(
    CONFIG["role_families_file"],
    header=True,
    inferSchema=True
)

print(f"Loaded {role_families_df.count()} role families")
print("\nRole Families by Sector:")
display(role_families_df.orderBy("sector_key", "family_key"))

In [0]:
# Load canonical roles
roles_df = spark.read.csv(
    CONFIG["canonical_roles_file"],
    header=True,
    inferSchema=True
)

print(f"Loaded {roles_df.count()} canonical roles")
print("\nRoles by Sector:")
display(
    roles_df.groupBy("sector_key").count()
    .orderBy(F.desc("count"))
)

In [0]:
# Load canonical skills
skills_df = spark.read.csv(
    CONFIG["canonical_skills_file"],
    header=True,
    inferSchema=True
)

print(f"Loaded {skills_df.count()} canonical skills")
print("\nSkills by Category and Sector:")
display(
    skills_df.groupBy("skill_category", "sector_key")
    .count()
    .orderBy("skill_category", F.desc("count"))
)

In [0]:
# Build role lookup from aliases
# This replaces the hardcoded dictionary in semantic_role_map

from pyspark.sql.functions import explode, split

# Explode aliases into individual entries
role_lookup_df = roles_df.select(
    "canonical_role",
    "family_key",
    "sector_key",
    "seniority",
    explode(split("aliases", "\\|")).alias("alias")
).withColumn(
    "normalized_alias", F.lower(F.trim(F.col("alias")))
)

print(f"Built role lookup with {role_lookup_df.count()} alias entries")
print("\nSample role aliases:")
display(role_lookup_df.limit(10))

In [0]:
# Build sector keyword lookup
# This replaces the hardcoded NAICS taxonomy in semantic_sector_normalize

from pyspark.sql.functions import explode, split

# Explode keywords into individual entries
sector_keyword_df = sectors_df.select(
    "sector_key",
    "sector_name",
    "parent_sector",
    "naics_code",
    explode(split("keywords", "\\|")).alias("keyword")
).withColumn(
    "normalized_keyword", F.lower(F.trim(F.col("keyword")))
)

print(f"Built sector keyword lookup with {sector_keyword_df.count()} keyword entries")
print("\nSample sector keywords:")
display(sector_keyword_df.limit(10))

In [0]:
# Build skill lookup from aliases
# This replaces hardcoded skill catalogs

from pyspark.sql.functions import explode, split, coalesce, lit

# Explode aliases into individual entries
skill_lookup_df = skills_df.select(
    "canonical_skill",
    "skill_category",
    coalesce("sector_key", lit("CROSS_SECTOR")).alias("sector_key"),
    explode(split("aliases", "\\|")).alias("alias")
).withColumn(
    "normalized_alias", F.lower(F.trim(F.col("alias")))
)

print(f"Built skill lookup with {skill_lookup_df.count()} alias entries")
print("\nSkills by Sector:")
display(
    skill_lookup_df.groupBy("sector_key").count()
    .orderBy(F.desc("count"))
)

## How to Use in Intermediate Notebooks

### Instead of this (current):
```python
# inter_role_map (HARDCODED)
sample_dict = [
    ("Software Engineer", "software engineer", "exact", 1.0),
    ("Software Engineer", "swe", "alias", 0.95),
    ...
]
```

### Do this:
```python
# Load from metadata
role_lookup_df = spark.read.csv(f"{METADATA_BASE}/canonical_roles.csv", header=True)
role_dict_df = role_lookup_df.select(
    "canonical_role",
    explode(split("aliases", "\\|")).alias("alias")
)
```

### Key Changes:
1. **inter_sector_normalize**: Load `sectors.csv` instead of hardcoded NAICS samples
2. **inter_role_map**: Load `canonical_roles.csv` instead of hardcoded role dictionary
3. **inter_skill_catalog_sync**: Load `canonical_skills.csv` instead of hardcoded skill lists

### Result:
* **Technology** is just one sector in the taxonomy
* **Hospitality**, **Healthcare**, **Finance**, **Retail** are equally represented
* Adding new sectors = adding rows to CSV files, not rewriting Python code
* Your Knowledge Graph will be sector-agnostic from day one